# Day 3: Machine Learning

Topics covered:
- Supervised classification: Decision Tree on student performance
- Model evaluation: Accuracy, Confusion Matrix, Classification Report
- Decision tree visualization and depth comparison (max_depth 3, 4, 5)
- Supervised regression: Simple and Multiple Linear Regression on cricket match scores
- Hardware engineering regression and unsupervised K-Means clustering with PCA
- Polynomial regression for non-linear trends
- Simple Neural Network with Keras
- SQLite database creation, queries, and Pandas integration


## 1. Decision Tree Classification - Student Performance


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
csv_path = "student_performance.csv"
if not os.path.exists(csv_path) and os.path.exists("BOOTCAMP ON AI_Assignments/student_performance.csv"):
    csv_path = "BOOTCAMP ON AI_Assignments/student_performance.csv"
if os.path.exists(csv_path):
    df_students = pd.read_csv(csv_path)
else:
    np.random.seed(42)
    n = 100
    sh = np.round(np.random.uniform(1.0, 10.0, n), 1)
    att = np.random.randint(45, 100, n)
    asn = np.random.randint(1, 11, n)
    score = (sh * 0.4) + (att * 0.4) + (asn * 2.0)
    res = (score >= 45).astype(int)
    df_students = pd.DataFrame({"study_hours": sh, "attendance": att, "assignments": asn, "result": res})
display(df_students.head())
print("Average metrics grouped by result:")
print(df_students.groupby("result")[["study_hours", "attendance", "assignments"]].mean())


## 2. Train-Test Split and Model Training


In [ ]:
features = ["study_hours", "attendance", "assignments"]
X = df_students[features]
y = df_students["result"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(X_train, y_train)
y_pred = dt_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Fail (0)", "Pass (1)"]))


## 3. Visualizing the Decision Tree


In [ ]:
plt.figure(figsize=(15, 7), dpi=120)
plot_tree(
    dt_model,
    feature_names=features,
    class_names=["Fail (0)", "Pass (1)"],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Classifier (max_depth=3)")
plt.tight_layout()
plt.show()


## 4. Testing with New Student Data


In [ ]:
sample_candidates = pd.DataFrame([
    {"Name": "Aarav Sharma", "study_hours": 8.0, "attendance": 92, "assignments": 9},
    {"Name": "Sneha Patel",   "study_hours": 2.0, "attendance": 50, "assignments": 3},
    {"Name": "Rohan Gupta",   "study_hours": 5.5, "attendance": 78, "assignments": 7},
    {"Name": "Priya Singh",   "study_hours": 3.0, "attendance": 88, "assignments": 5}
])
preds = dt_model.predict(sample_candidates[features])
probs = dt_model.predict_proba(sample_candidates[features])
sample_candidates["Predicted Result"] = ["Pass" if p == 1 else "Fail" for p in preds]
sample_candidates["Confidence"] = [f"{probs[i][preds[i]]*100:.1f}%" for i in range(len(preds))]
display(sample_candidates[["Name", "study_hours", "attendance", "assignments", "Predicted Result", "Confidence"]])


## 5. Comparing Tree Depths (3, 4, 5)


In [ ]:
depths = [3, 4, 5]
comparison_records = []
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, clf.predict(X_train))
    test_acc = accuracy_score(y_test, clf.predict(X_test))
    comparison_records.append({
        "Max Depth": d,
        "Train Accuracy (%)": round(train_acc * 100, 2),
        "Test Accuracy (%)": round(test_acc * 100, 2),
        "Total Leaves": clf.get_n_leaves()
    })
comparison_df = pd.DataFrame(comparison_records)
display(comparison_df)
plt.figure(figsize=(8, 4.5), dpi=100)
plt.plot(depths, comparison_df["Train Accuracy (%)"], marker='o', lw=2.5, color='#2563eb', label="Training Accuracy")
plt.plot(depths, comparison_df["Test Accuracy (%)"], marker='s', lw=2.5, color='#16a34a', linestyle='--', label="Testing Accuracy")
plt.title("Train vs. Test Accuracy Across Depths")
plt.xlabel("Max Depth")
plt.ylabel("Accuracy (%)")
plt.xticks(depths)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Linear Regression - Match Score Prediction


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
np.random.seed(42)
n_matches = 800
t1_runs = np.random.randint(120, 240, n_matches)
t1_wickets = np.random.randint(2, 10, n_matches)
t2_wickets = np.random.randint(2, 10, n_matches)
t2_runs = (t1_runs * 0.85 + (10 - t2_wickets) * 4 + np.random.normal(0, 12, n_matches)).astype(int)
df_matches = pd.DataFrame({
    "team1_runs": t1_runs,
    "team1_wickets": t1_wickets,
    "team2_wickets": t2_wickets,
    "team2_runs": t2_runs
})
display(df_matches.head())
X_simple = df_matches[["team1_runs"]]
y_target = df_matches["team2_runs"]
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_simple, y_target, test_size=0.2, random_state=42)
simple_lr = LinearRegression()
simple_lr.fit(X_train_s, y_train_s)
y_pred_s = simple_lr.predict(X_test_s)
print(f"Simple LR Equation : team2_runs = {simple_lr.coef_[0]:.3f} * team1_runs + ({simple_lr.intercept_:.2f})")
print(f"Simple LR R2 Score : {r2_score(y_test_s, y_pred_s):.4f} | MAE: {mean_absolute_error(y_test_s, y_pred_s):.2f}")


## 7. Multiple Linear Regression


In [ ]:
features_multi = ["team1_runs", "team1_wickets", "team2_wickets"]
X_multi = df_matches[features_multi]
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_target, test_size=0.2, random_state=42)
multi_lr = LinearRegression()
multi_lr.fit(X_train_m, y_train_m)
y_pred_m = multi_lr.predict(X_test_m)
print(f"Multiple LR R2 Score: {r2_score(y_test_m, y_pred_m):.4f} | MAE: {mean_absolute_error(y_test_m, y_pred_m):.2f}")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=100)
ax1.scatter(X_test_s, y_test_s, color='#3b82f6', alpha=0.6, edgecolors='k')
line_x = np.linspace(X_test_s.min().values[0], X_test_s.max().values[0], 100).reshape(-1, 1)
line_y = simple_lr.predict(line_x)
ax1.plot(line_x, line_y, color='crimson', lw=2.5)
ax1.set_title("Simple Linear Regression Line")
ax1.set_xlabel("Team 1 Runs")
ax1.set_ylabel("Team 2 Runs")
ax1.grid(True, linestyle="--", alpha=0.5)
ax2.scatter(y_test_m, y_pred_m, color='#10b981', alpha=0.6, edgecolors='k')
diag = [min(y_test_m.min(), y_pred_m.min()), max(y_test_m.max(), y_pred_m.max())]
ax2.plot(diag, diag, color='crimson', linestyle='--', lw=2)
ax2.set_title("Multiple LR: Actual vs Predicted")
ax2.set_xlabel("Actual Runs")
ax2.set_ylabel("Predicted Runs")
ax2.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## 8. Hardware Regression and K-Means Clustering


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
np.random.seed(42)
n_chips = 300
clock_speed = np.round(np.random.uniform(1.8, 5.2, n_chips), 2)
voltage = np.round(np.random.uniform(0.75, 1.45, n_chips), 3)
core_count = np.random.choice([4, 6, 8, 12, 16, 24, 32], n_chips)
tdp_power = np.round(22 * (voltage**2) * clock_speed * (core_count**0.7) + np.random.normal(0, 4, n_chips), 1)
hw_df = pd.DataFrame({
    "clock_speed_ghz": clock_speed,
    "core_voltage_v": voltage,
    "core_count": core_count,
    "tdp_power_watts": tdp_power
})
hw_X = hw_df[["clock_speed_ghz", "core_voltage_v", "core_count"]]
hw_y = hw_df["tdp_power_watts"]
hw_X_train, hw_X_test, hw_y_train, hw_y_test = train_test_split(hw_X, hw_y, test_size=0.2, random_state=42)
hw_model = LinearRegression().fit(hw_X_train, hw_y_train)
print(f"Hardware TDP Model R2: {hw_model.score(hw_X_test, hw_y_test):.4f}")
scaler = StandardScaler()
hw_scaled = scaler.fit_transform(hw_df)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
hw_df["Cluster"] = kmeans.fit_predict(hw_scaled)
pca = PCA(n_components=2)
hw_pca = pca.fit_transform(hw_scaled)
hw_df["PCA1"] = hw_pca[:, 0]
hw_df["PCA2"] = hw_pca[:, 1]
plt.figure(figsize=(8, 4.5), dpi=100)
scatter = plt.scatter(hw_df["PCA1"], hw_df["PCA2"], c=hw_df["Cluster"], cmap="viridis", edgecolors='k', s=50, alpha=0.85)
plt.colorbar(scatter, label="Cluster")
plt.title("K-Means Hardware Clustering (PCA 2D)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## 9. Polynomial Regression


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
np.random.seed(42)
X_curve = np.linspace(-3, 3, 100).reshape(-1, 1)
y_curve = 0.6 * (X_curve**3) - 1.2 * (X_curve**2) + 2.5 * X_curve + np.random.normal(0, 1.5, (100, 1))
poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X_curve)
linear_reg = LinearRegression().fit(X_curve, y_curve)
poly_reg = LinearRegression().fit(X_poly, y_curve)
plt.figure(figsize=(9, 4.5), dpi=100)
plt.scatter(X_curve, y_curve, color='#64748b', alpha=0.7, edgecolors='k', label='Data')
plt.plot(X_curve, linear_reg.predict(X_curve), color='crimson', lw=2, linestyle='--', label=f'Linear (R2={r2_score(y_curve, linear_reg.predict(X_curve)):.2f})')
plt.plot(X_curve, poly_reg.predict(X_poly), color='#2563eb', lw=2.5, label=f'Polynomial deg 3 (R2={r2_score(y_curve, poly_reg.predict(X_poly)):.2f})')
plt.title("Linear vs. Polynomial Regression")
plt.xlabel("X")
plt.ylabel("y")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


## 10. Neural Network with Keras


In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense
    ann_model = Sequential([
        Dense(16, activation='relu', input_shape=(1,)),
        Dense(16, activation='relu'),
        Dense(1, activation='linear')
    ])
    ann_model.compile(optimizer='adam', loss='mse')
    ann_model.summary()
    history = ann_model.fit(X_curve, y_curve, epochs=60, verbose=0)
    print("Training finished. Final MSE Loss:", history.history['loss'][-1])
    ann_pred = ann_model.predict(X_curve, verbose=0)
    plt.figure(figsize=(8, 4), dpi=100)
    plt.scatter(X_curve, y_curve, color='#94a3b8', alpha=0.6, label='Data')
    plt.plot(X_curve, ann_pred, color='#7c3aed', lw=2.5, label='ANN Predictions')
    plt.title("Neural Network Non-Linear Approximation")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("TensorFlow not installed in this environment.")


## 11. SQLite Database Operations


In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute('''
CREATE TABLE band_catalog (
    band_id TEXT PRIMARY KEY,
    band_name TEXT,
    temperature_celsius REAL,
    radiance_lmax REAL
)
''')
bands_data = [
    ("BAND2", "Green", 17.50, 14.85),
    ("BAND3", "Red",   18.20, 16.50),
    ("BAND4", "NIR",   19.80, 18.90),
    ("BAND5", "SWIR",  21.10, 12.30)
]
cursor.executemany("INSERT INTO band_catalog VALUES (?, ?, ?, ?)", bands_data)
conn.commit()
df_bands = pd.read_sql_query("SELECT * FROM band_catalog", conn)
display(df_bands)
cursor.execute("SELECT AVG(temperature_celsius), MAX(radiance_lmax) FROM band_catalog")
avg_temp, max_rad = cursor.fetchone()
print(f"Average Temperature: {avg_temp:.2f} C | Max Radiance: {max_rad}")
conn.close()
